In [1]:
import os
os.environ['USE_PYGEOS'] = '0'

import numpy
import pandas
from datetime import datetime, timedelta
import pytz
import geopandas
import dask_geopandas
import shapely
from shapely import wkb, wkt
import pyproj

import subprocess
import shutil
from glob import glob
import json

from functools import partial
from multiprocessing import Pool
from random import shuffle

import matplotlib.pyplot as plt

In [2]:
raw_geoparquet_path = '/data/raster/overture/base_raw'
processed_geoparquet_path = '/data/raster/overture/base_processed'
files = glob(os.path.join(raw_geoparquet_path, '**/*.parquet'))
len(files)

143

In [3]:
class NpEncoder(json.JSONEncoder):
    # To avoid json type errors
    def default(self, obj):
        if isinstance(obj, numpy.integer):
            return int(obj)
        if isinstance(obj, numpy.floating):
            return float(obj)
        if isinstance(obj, numpy.ndarray):
            return obj.tolist()
        return super(NpEncoder, self).default(obj)
        
def _split_names(names_dict, cols):
    """Split names dictionaries into those we want to extract as columns and those we want to keep as json."""
    explode_cols = ['primary']
    explode_names = {x: names_dict[x] for x in names_dict if x in explode_cols}
    other_names = {x: names_dict[x] for x in names_dict if x not in explode_cols}
    
    return explode_names, other_names

def _names_to_columns(gdf, cols):
    """Extract the primary name from names and move the remaining ones to other_names."""
    if len(gdf)>0:
        # Split names json dictionaries into those we want to extract as columns and those we want to keep as json.
        gdf[['explode_names', 'other_names']] = gdf['names'].apply(lambda x: _split_names(x, cols)).to_list()
        
        # Convert explode_names to columns
        gdf_tmp = pandas.DataFrame(gdf['explode_names'].to_list())
        other_cols = sorted(set(gdf.columns) - set(gdf_tmp.columns) - set(['names', 'explode_names']))
        gdf = pandas.concat([gdf_tmp, gdf[other_cols]], axis=1)
        
        # Dumping other_names to json string so that the column can be saved efficiently in parquet
        gdf['other_names'] = gdf['other_names'].apply(lambda x: json.dumps(x, cls=NpEncoder))
        
    return geopandas.GeoDataFrame(gdf)

In [4]:
cols = []
for file in files:
    gdf = geopandas.read_parquet(
        file,
        filters=[('version', '!=', 0)]
    )
    if len(gdf)>0:
        print(file)
    cols.extend(list(gdf.columns))
cols = sorted(set(cols))

exclude_cols = [
    'bbox',
    'version', 
    'cartography',
    'names',
]

include_cols = [
    'id',
    'update_time',
    'type', 
    'subtype', 
    'class',
    'name',
    'sources',
    'wikidata',
    'level',
    'elevation',
    'surface',
    'is_salt',
    'is_intermittent',
    'source_tags',
    'geometry',
]
print('Removing these columns:', set(exclude_cols))
print('Adding these columns:  ', (set(exclude_cols) | set(include_cols)) - set(cols))
assert len(set(cols) - (set(exclude_cols) | set(include_cols)))==0

Removing these columns: {'cartography', 'bbox', 'version', 'names'}
Adding these columns:   {'name', 'type'}


In [5]:
for file in files:
    file = file.replace('\\', '/')
    print(file)
    gdf = geopandas.read_parquet(file)
   
    # Get the hive-style "type" column value
    gdf['type'] = os.path.split(os.path.split(file)[0])[1].split('type=')[1]
    
    if 'names' in gdf.columns:
        # Replace None with empty dictionary
        gdf[['names']]=gdf[['names']].map(lambda x: {} if pandas.isnull(x) else x)
        
        # Extract the primary name
        gdf = _names_to_columns(gdf, 'names')
        gdf = gdf.rename(columns={'primary': 'name'})
    
    # Common set of columns for all parquet files
    for col in exclude_cols:
        if col in gdf.columns:
            del gdf[col]
    for col in include_cols:
        if col not in gdf.columns:
            gdf[col] = None
    gdf = gdf[include_cols]
    
    # Dumping some columns to json string so that the column can be saved efficiently in parquet
    gdf['sources'] = gdf['sources'].apply(lambda x: json.dumps(x, cls=NpEncoder))
    gdf['source_tags'] = gdf['source_tags'].apply(lambda x: json.dumps(x, cls=NpEncoder))
    
    # Timestamp string to datetime
    gdf['update_time'] = pandas.to_datetime(gdf['update_time'])
    
    # Write to file
    out_path = os.path.join(
        processed_geoparquet_path, 
        file.split(raw_geoparquet_path+'/')[1]
    ).replace('\\', '/')
    out_dir = os.path.split(out_path)[0]
    if not os.path.exists(out_dir):
        os.makedirs(out_dir)
    gdf.to_parquet(out_path)
    

/data/raster/overture/base_raw/type=infrastructure/part-00000-df7630cc-540a-4931-b01e-6f33b054dd61-c000.zstd.parquet
/data/raster/overture/base_raw/type=infrastructure/part-00001-df7630cc-540a-4931-b01e-6f33b054dd61-c000.zstd.parquet
/data/raster/overture/base_raw/type=infrastructure/part-00002-df7630cc-540a-4931-b01e-6f33b054dd61-c000.zstd.parquet
/data/raster/overture/base_raw/type=infrastructure/part-00003-df7630cc-540a-4931-b01e-6f33b054dd61-c000.zstd.parquet
/data/raster/overture/base_raw/type=infrastructure/part-00004-df7630cc-540a-4931-b01e-6f33b054dd61-c000.zstd.parquet
/data/raster/overture/base_raw/type=infrastructure/part-00005-df7630cc-540a-4931-b01e-6f33b054dd61-c000.zstd.parquet
/data/raster/overture/base_raw/type=infrastructure/part-00006-df7630cc-540a-4931-b01e-6f33b054dd61-c000.zstd.parquet
/data/raster/overture/base_raw/type=land/part-00000-70f3f59e-69da-4f64-86c7-43473311dfdd-c000.zstd.parquet
/data/raster/overture/base_raw/type=land/part-00001-70f3f59e-69da-4f64-86c

In [6]:
dask_geopandas.read_parquet(processed_geoparquet_path, split_row_groups=False)

,id,update_time,type,subtype,class,name,sources,wikidata,level,elevation,surface,is_salt,is_intermittent,source_tags,geometry
npartitions=143,,,,,,,,,,,,,,,
,object,"datetime64[ns, UTC]",category[known],object,object,object,object,object,float64,object,object,object,object,object,geometry
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
